In [5]:
from pathlib import Path
import pandas as pd

GROUPS = {
    "donors":   {"src": Path("../data/donors_processed"),   "dst": Path("../data/donors_clono")},
    "patients": {"src": Path("../data/patients_processed"), "dst": Path("../data/patients_clono")},
}
KEY = ["CDR3_sequence", "TRBV", "TRBJ"]          # same clonotype key used to build counts/

for group, pth in GROUPS.items():
    src, dst = pth["src"], pth["dst"]
    (dst / "umi").mkdir(exist_ok=True)
    subjects = sorted(f.stem for f in (dst / "counts").glob("*.csv"))
    sizes = []

    for s in subjects:
        raw = pd.read_parquet(src / f"{s}_raw_IN.parquet", columns=KEY + ["day", "Count", "UMI"])
        for k in ["TRBV", "TRBJ"]:
            raw[k] = raw[k].astype(str)            # matches how sequences/ was written (NaN -> 'nan')
        n_nan = int(raw["UMI"].isna().sum())

        seq = pd.read_csv(dst / "sequences" / f"{s}_sequences.csv",
                          dtype={"clono": "int32", "CDR3_sequence": str, "TRBV": str, "TRBJ": str},
                          keep_default_na=False)
        m = raw.merge(seq, on=KEY, how="left", validate="many_to_one")
        assert m["clono"].notna().all(), f"{group}/{s}: raw rows without a clono id"

        agg = (m.groupby(["day", "clono"], as_index=False)
                 .agg(umi=("UMI", "sum"), count_raw=("Count", "sum")))
        agg["day"] = agg["day"].astype("int16")
        agg["clono"] = agg["clono"].astype("int32")

        # exactly the rows of the counts file, with a check that the counts agree
        cnt = pd.read_csv(dst / "counts" / f"{s}.csv", dtype={"day": "int16", "clono": "int32"})
        out = cnt.merge(agg, on=["day", "clono"], how="left", validate="one_to_one")
        assert out["umi"].notna().all(), f"{group}/{s}: counts rows missing in the raw data"
        assert (out["count"] == out["count_raw"]).all(), f"{group}/{s}: counts differ from raw"

        out = out[["day", "clono", "umi"]].astype({"umi": "int64"}).sort_values(["day", "clono"])
        out.to_csv(dst / "umi" / f"{s}.csv", index=False)

        u = out.groupby("day")["umi"].sum().rename("U_t").reset_index()
        u.insert(0, "patient", s)
        sizes.append(u)
        print(f"{group:8s} {s:22s} {len(out):>9,d} rows | UMI==0 rows: {int((out['umi'] == 0).sum())} "
              f"| UMI missing in raw: {n_nan}")
        del raw, seq, m, agg, cnt, out

    pd.concat(sizes, ignore_index=True).to_csv(dst / "samples_sizes_UMI.csv", index=False)

donors   HD106_SEQTR            3,803,918 rows | UMI==0 rows: 0 | UMI missing in raw: 0


AssertionError: donors/HD197_SEQTR: raw rows without a clono id

In [6]:
s, dst, src = "HD197_SEQTR", Path("../data/donors_clono"), Path("../data/donors_processed")
raw = pd.read_parquet(src / f"{s}_raw_IN.parquet", columns=KEY + ["day", "Count", "UMI"])
for k in ["TRBV", "TRBJ"]:
    raw[k] = raw[k].astype(str)
seq = pd.read_csv(dst / "sequences" / f"{s}_sequences.csv",
                  dtype={"clono": "int32", "CDR3_sequence": str, "TRBV": str, "TRBJ": str},
                  keep_default_na=False)
m = raw.merge(seq, on=KEY, how="left", validate="many_to_one")
bad = m[m["clono"].isna()]

print("unmatched rows:", len(bad), "of", len(m))
print("by day:", bad["day"].value_counts().sort_index().to_dict())
print("Count of unmatched:", bad["Count"].describe().to_dict())
print("Count == 0:", int((bad["Count"] == 0).sum()), "| Count NaN:", int(bad["Count"].isna().sum()))
print("CDR3 missing:", int(bad["CDR3_sequence"].isna().sum()),
      "| TRBV 'nan':", int((bad["TRBV"] == "nan").sum()), "| TRBJ 'nan':", int((bad["TRBJ"] == "nan").sum()))
print("map: empty CDR3:", int((seq["CDR3_sequence"] == "").sum()),
      "| sizes raw/map unique keys:", raw[KEY].drop_duplicates().shape[0], len(seq))

unmatched rows: 2 of 4176400
by day: {46: 1, 82: 1}
Count of unmatched: {'count': 2.0, 'mean': 36.0, 'std': 39.59797974644666, 'min': 8.0, '25%': 22.0, '50%': 36.0, '75%': 50.0, 'max': 64.0}
Count == 0: 0 | Count NaN: 0
CDR3 missing: 2 | TRBV 'nan': 0 | TRBJ 'nan': 0
map: empty CDR3: 2 | sizes raw/map unique keys: 3816358 3816358
